In [1]:
import pandas as pd

nav = pd.read_csv("../data/raw/02_nav_history.csv")

In [2]:
# Date conversion
nav['date'] = pd.to_datetime(nav['date'])

# Sort
nav = nav.sort_values(['amfi_code', 'date'])

# Remove duplicates
nav = nav.drop_duplicates()

# NAV > 0
nav = nav[nav['nav'] > 0]

# Forward fill NAV
nav['nav'] = nav.groupby('amfi_code')['nav'].ffill()

In [4]:
nav.to_csv("../data/processd/clean_nav_history.csv", index=False)

In [5]:
tx = pd.read_csv("../data/raw/08_investor_transactions.csv")
tx.columns

Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='object')

In [6]:
tx['transaction_type'] = (
    tx['transaction_type']
    .str.strip()
    .str.title()
)

In [8]:
tx.columns

Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='object')

In [9]:
tx = tx[tx['amount_inr'] > 0]

In [10]:
tx['kyc_status'].value_counts()

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

In [12]:
tx.to_csv("../data/processd/clean_transactions.csv", index=False)

In [13]:
perf = pd.read_csv("../data/raw/07_scheme_performance.csv")

In [14]:
perf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ra

In [15]:
perf['expense_ratio_pct'] = pd.to_numeric(
    perf['expense_ratio_pct'],
    errors='coerce'
)

In [17]:
perf[
    (perf['expense_ratio_pct'] < 0.1)
    |
    (perf['expense_ratio_pct'] > 2.5)
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [19]:
perf.to_csv(
    "../data/processd/clean_performance.csv",
    index=False
)

In [1]:
from sqlalchemy import create_engine

engine = create_engine(
    "sqlite:///../data/db/bluestock_mf.db"
)

print("Database Created")

Database Created


In [3]:
import pandas as pd

fund_master = pd.read_csv(
    "../data/raw/01_fund_master.csv"
)

fund_master = fund_master.drop_duplicates()

fund_master.to_csv(
    "../data/processd/clean_fund_master.csv",
    index=False
)

print("Fund Master Cleaned")

Fund Master Cleaned


In [4]:
aum = pd.read_csv(
    "../data/raw/03_aum_by_fund_house.csv"
)

aum = aum.drop_duplicates()

aum.to_csv(
    "../data/processd/clean_aum.csv",
    index=False
)

In [5]:
from sqlalchemy import create_engine

engine = create_engine(
    "sqlite:///../data/db/bluestock_mf.db"
)

print("Connected Successfully")

Connected Successfully


In [6]:
import pandas as pd

fund_master = pd.read_csv(
    "../data/processd/clean_fund_master.csv"
)

fund_master.to_sql(
    "dim_fund",
    engine,
    if_exists="replace",
    index=False
)

print("dim_fund loaded")

dim_fund loaded


In [8]:
tx = pd.read_csv(
    "../data/processd/clean_transactions.csv"
)

tx.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

print("fact_transactions loaded")

fact_transactions loaded


In [10]:
perf = pd.read_csv(
    "../data/processd/clean_performance.csv"
)

perf.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)

print("fact_performance loaded")

fact_performance loaded


In [12]:
aum = pd.read_csv(
    "../data/processd/clean_aum.csv"
)

aum.to_sql(
    "fact_aum",
    engine,
    if_exists="replace",
    index=False
)

print("fact_aum loaded")

fact_aum loaded


In [18]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/db/bluestock_mf.db")

tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

tables

,name
0,dim_fund
1,fact_transactions
2,fact_performance
3,fact_aum
4,fact_nav


In [16]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine(
    "sqlite:///../data/db/bluestock_mf.db"
)

nav = pd.read_csv(
    "../data/processd/clean_nav_history.csv"
)

print(nav.shape)

nav.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

print("fact_nav loaded successfully")

(46000, 3)
fact_nav loaded successfully


In [19]:
tables = [
    "dim_fund",
    "fact_nav",
    "fact_transactions",
    "fact_performance",
    "fact_aum"
]

for table in tables:
    result = pd.read_sql(
        f"SELECT COUNT(*) as rows FROM {table}",
        conn
    )
    print(table)
    print(result)
    print()

dim_fund
   rows
0    40

fact_nav
    rows
0  46000

fact_transactions
    rows
0  32778

fact_performance
   rows
0    40

fact_aum
   rows
0    90

